In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import time
import ydata_profiling as pp

print("Enhanced QB Decision Recommendation System (Fixed Version)")
print("=======================================================")
start_time = time.time()

# 1. DATA LOADING
print("\n1. LOADING DATA")
print("--------------")

datasets_info = [
    {
        'name': 'games',
        'path': '../data/raw/games.csv',
        'required': False,
        'description': 'Game metadata'
    },
    {
        'name': 'plays',
        'path': '../data/raw/plays.csv',
        'required': True,
        'description': 'Play-by-play data with game situations'
    },
    {
        'name': 'player_play',
        'path': '../data/raw/player_play.csv',
        'required': True,
        'description': 'Player participation in each play'
    },
    {
        'name': 'players',
        'path': '../data/raw/players.csv',
        'required': True,
        'description': 'Player attributes and information'
    },
    {
        'name': 'qb_features',
        'path': '../data/processed/qb_decision_features.csv',
        'required': False,
        'description': 'Preprocessed QB decision features'
    },
    {
        'name': 'receiver_features',
        'path': '../data/processed/receiver_target_features.csv',
        'required': True,
        'description': 'Preprocessed receiver target features'
    },
    {
        'name': 'player_features',
        'path': '../data/processed/player_features.csv',
        'required': False,
        'description': 'Preprocessed player physical features'
    },
    {
        'name': 'tracking_features',
        'path': '../data/processed/multi_week_player_tracking_features.csv',
        'required': False,
        'description': 'Player movement tracking features'
    }
]

datasets = {}
loading_errors = []

for ds in datasets_info:
    try:
        datasets[ds['name']] = pd.read_csv(ds['path'])
        print(f"✓ Loaded {ds['name']}: {len(datasets[ds['name']])} rows")
    except Exception as e:
        loading_errors.append(f"Could not load {ds['name']}: {str(e)}")
        print(f"✗ Error loading {ds['name']}: {e}")
        
        if ds['required']:
            print(f"  Creating minimal {ds['name']} dataset")
            
            if ds['name'] == 'plays':
                datasets[ds['name']] = pd.DataFrame({
                    'gameId': [2021090900, 2021090900],
                    'playId': [1, 2],
                    'down': [1, 3],
                    'yardsToGo': [10, 5],
                    'absoluteYardlineNumber': [75, 25],
                    'quarter': [1, 2]
                })
            elif ds['name'] == 'player_play':
                datasets[ds['name']] = pd.DataFrame({
                    'gameId': [2021090900, 2021090900],
                    'playId': [1, 2],
                    'nflId': [1000, 1001],
                    'wasTargettedReceiver': [True, False],
                    'wasRunningRoute': [True, True]
                })
            elif ds['name'] == 'players':
                datasets[ds['name']] = pd.DataFrame({
                    'nflId': [1000, 1001],
                    'displayName': ['Player A', 'Player B'],
                    'position': ['WR', 'TE'],
                    'height': ['6-2', '6-5'],
                    'weight': [200, 250]
                })
            elif ds['name'] == 'receiver_features':
                datasets[ds['name']] = pd.DataFrame({
                    'gameId': [2021090900, 2021090900],
                    'playId': [1, 2],
                    'nflId': [1000, 1001],
                    'was_targeted': [1, 0],
                    'route_ran': ['SLANT', 'GO'],
                    'coverage_assignment': ['MAN', 'ZONE'],
                    'down': [1, 3],
                    'yards_to_go': [10, 5],
                    'yards_from_goal': [75, 25],
                    'quarter': [1, 2]
                })
        else:
            datasets[ds['name']] = pd.DataFrame()

if loading_errors:
    print("\nWARNING: Some datasets could not be loaded properly. The model may have limited functionality.")
    print(f"Missing {len(loading_errors)} of {len(datasets_info)} datasets")
else:
    print("\nAll datasets loaded successfully!")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enhanced QB Decision Recommendation System (Fixed Version)

1. LOADING DATA
--------------
✓ Loaded games: 136 rows
✓ Loaded plays: 16124 rows
✓ Loaded player_play: 354727 rows
✓ Loaded players: 1697 rows
✓ Loaded qb_features: 9736 rows
✓ Loaded receiver_features: 46566 rows
✓ Loaded player_features: 1697 rows
✓ Loaded tracking_features: 354728 rows

All datasets loaded successfully!


In [3]:
# 2. DATA PREPROCESSING AND FEATURE ENGINEERING
print("\n2. DATA PREPROCESSING AND FEATURE ENGINEERING")
print("-------------------------------------------")

print("Preparing feature set...")
if 'receiver_features' in datasets and not datasets['receiver_features'].empty:
    model_df = datasets['receiver_features'].copy()
    print(f"Base feature set: {len(model_df)} receiver target examples")
else:
    print("ERROR: No receiver_features data available! Creating minimal dataset.")
    model_df = pd.DataFrame({
        'gameId': [2021090900, 2021090900],
        'playId': [1, 2],
        'nflId': [1000, 1001],
        'was_targeted': [1, 0],
        'route_ran': ['SLANT', 'GO'],
        'down': [1, 3],
        'yards_to_go': [10, 5],
        'yards_from_goal': [75, 25]
    })

if 'player_features' in datasets and not datasets['player_features'].empty:
    physical_attributes = [
        'height_inches', 'weight', 'receiver_type', 'bmi', 'height_vs_db',
        'position_group', 'age', 'is_receiver'
    ]

    available_attributes = [col for col in physical_attributes if col in datasets['player_features'].columns]

    if available_attributes:
        model_df = pd.merge(
            model_df,
            datasets['player_features'][['nflId'] + available_attributes],
            on='nflId',
            how='left'
        )
        print(f"Added player physical attributes: {available_attributes}")
else:
    print("No player physical attributes available")

if 'qb_features' in datasets and not datasets['qb_features'].empty:
    qb_features_list = [
        'pass_length', 'time_to_throw', 'pressure', 'expected_points',
        'time_in_pocket', 'play_action', 'shotgun', 'empty',
        'man_coverage', 'zone_coverage',
        'epa_success', 'target_depth', 'target_width', 'target_distance'
    ]

    numeric_qb_cols = []
    for col in qb_features_list:
        if col in datasets['qb_features'].columns:
            if np.issubdtype(datasets['qb_features'][col].dtype, np.number) or datasets['qb_features'][col].dtype == bool:
                numeric_qb_cols.append(col)
            else:
                print(f"  Skipping non-numeric QB feature: {col} (type: {datasets['qb_features'][col].dtype})")

    if numeric_qb_cols:
        qb_features_unique = datasets['qb_features'][['gameId', 'playId'] + numeric_qb_cols].drop_duplicates(['gameId', 'playId'])

        model_df = pd.merge(
            model_df,
            qb_features_unique,
            on=['gameId', 'playId'],
            how='left'
        )
        print(f"Added QB features: {numeric_qb_cols}")
else:
    print("No QB features available")

if 'tracking_features' in datasets and not datasets['tracking_features'].empty:
    tracking_features_list = [
        'route_directness', 'direction_changes', 's_mean', 's_max',
        'movement_alignment_mean', 'dis_sum',
        's_mean_POST_SNAP', 's_max_POST_SNAP'
    ]

    numeric_tracking_cols = []
    for col in tracking_features_list:
        if col in datasets['tracking_features'].columns:
            if np.issubdtype(datasets['tracking_features'][col].dtype, np.number):
                numeric_tracking_cols.append(col)
            else:
                print(f"  Skipping non-numeric tracking feature: {col} (type: {datasets['tracking_features'][col].dtype})")

    if numeric_tracking_cols:
        model_df = pd.merge(
            model_df,
            datasets['tracking_features'][['gameId', 'playId', 'nflId'] + numeric_tracking_cols],
            on=['gameId', 'playId', 'nflId'],
            how='left'
        )
        print(f"Added tracking features: {numeric_tracking_cols}")
else:
    print("No tracking features available")

print("Creating derived features...")

available_columns = model_df.columns.tolist()

if 'route_ran' in available_columns:
    route_categories = {
        'FLAT': 'short',
        'HITCH': 'short',
        'OUT': 'short',
        'SCREEN': 'short',
        'SLANT': 'short',
        'CROSS': 'medium',
        'DIG': 'medium',
        'IN': 'medium',
        'POST': 'deep',
        'CORNER': 'deep',
        'GO': 'deep'
    }
    model_df['route_category'] = model_df['route_ran'].map(route_categories)
    print("  Added route_category")

if all(col in available_columns for col in ['down', 'yards_to_go']):
    model_df['third_down_critical'] = ((model_df['down'] == 3) &
                                     (model_df['yards_to_go'] >= 3) &
                                     (model_df['yards_to_go'] <= 7)).astype(int)
    print("  Added third_down_critical")

if all(col in available_columns for col in ['yards_from_goal', 'down']):
    model_df['red_zone'] = (model_df['yards_from_goal'] <= 20).astype(int)
    model_df['redzone_scoring_opportunity'] = ((model_df['red_zone'] == 1) &
                                             (model_df['down'] >= 3)).astype(int)
    print("  Added redzone_scoring_opportunity")

if all(col in available_columns for col in ['pressure', 'time_to_throw']):
    model_df['quick_throw_under_pressure'] = ((model_df['pressure'] == 1) &
                                            (model_df['time_to_throw'] < 2.5)).astype(int)
    print("  Added quick_throw_under_pressure")

if all(col in available_columns for col in ['route_category', 'yards_to_go']):
    def depth_match(row):
        if pd.isna(row['route_category']):
            return np.nan

        if row['yards_to_go'] <= 3 and row['route_category'] == 'short':
            return 1
        elif 3 < row['yards_to_go'] <= 10 and row['route_category'] == 'medium':
            return 1
        elif row['yards_to_go'] > 10 and row['route_category'] == 'deep':
            return 1
        else:
            return 0

    model_df['route_depth_match'] = model_df.apply(depth_match, axis=1)
    print("  Added route_depth_match")

if 'age' in available_columns:
    def get_experience_level(age):
        if pd.isna(age):
            return 'unknown'
        elif age <= 24:
            return 'rookie_young'
        elif age <= 28:
            return 'prime'
        elif age <= 32:
            return 'veteran'
        else:
            return 'senior'

    model_df['experience_level'] = model_df['age'].apply(get_experience_level)

    if 'third_down_critical' in available_columns:
        model_df['veteran_in_critical'] = (
            (model_df['experience_level'].isin(['veteran', 'senior'])) &
            (model_df['third_down_critical'] == 1)
        ).astype(int)
        print("  Added veteran_in_critical")

    print("  Added experience_level")

if all(col in available_columns for col in ['coverage_assignment', 'height_inches']):
    model_df['height_advantage_in_man'] = (
        (model_df['coverage_assignment'] == 'MAN') &
        (model_df['height_vs_db'] > 2)
    ).astype(int)
    print("  Added height_advantage_in_man")

if all(col in available_columns for col in ['shotgun', 'route_category']):
    model_df['shotgun_deep_route'] = (
        (model_df['shotgun'] == 1) &
        (model_df['route_category'] == 'deep')
    ).astype(int)
    print("  Added shotgun_deep_route")

if all(col in available_columns for col in ['s_max', 'nearest_defender_dist']):
    model_df['speed_with_separation'] = (
        (model_df['s_max'] > model_df['s_max'].median()) &
        (model_df['nearest_defender_dist'] > 3)
    ).astype(int)
    print("  Added speed_with_separation")

if all(col in available_columns for col in ['target_depth', 'target_width']):
    model_df['target_distance_calculated'] = np.sqrt(
        model_df['target_depth']**2 + model_df['target_width']**2
    )
    print("  Added target_distance_calculated")

print("Handling missing values...")

model_df['receiver_type'] = model_df['receiver_type'].fillna('not_receiver')

numeric_cols = model_df.select_dtypes(include=['float64', 'int64']).columns
model_df[numeric_cols] = model_df[numeric_cols].fillna(model_df[numeric_cols].median())

categorical_cols = model_df.select_dtypes(include=['object']).columns
model_df[categorical_cols] = model_df[categorical_cols].fillna('unknown')

print(f"Final dataset shape: {model_df.shape}")


2. DATA PREPROCESSING AND FEATURE ENGINEERING
-------------------------------------------
Preparing feature set...
Base feature set: 46566 receiver target examples
Added player physical attributes: ['height_inches', 'weight', 'receiver_type', 'bmi', 'height_vs_db', 'position_group', 'age', 'is_receiver']
  Skipping non-numeric QB feature: pressure (type: object)
Added QB features: ['pass_length', 'time_to_throw', 'expected_points', 'time_in_pocket', 'play_action', 'shotgun', 'empty', 'epa_success', 'target_depth', 'target_width', 'target_distance']
Added tracking features: ['route_directness', 'direction_changes', 's_mean', 's_max', 'movement_alignment_mean', 'dis_sum', 's_mean_POST_SNAP', 's_max_POST_SNAP']
Creating derived features...
  Added route_category
  Added third_down_critical
  Added redzone_scoring_opportunity
  Added experience_level
  Added height_advantage_in_man
  Added target_distance_calculated
Handling missing values...
Final dataset shape: (46566, 44)


In [4]:
# 3. PREPARE TRAINING DATA
print("\n3. PREPARE TRAINING DATA")
print("----------------------")

model_df = model_df.drop_duplicates()

if 'was_targeted' not in model_df.columns:
    print("ERROR: Target variable 'was_targeted' not found. Creating synthetic target.")
    model_df['was_targeted'] = np.random.choice([0, 1], len(model_df), p=[0.8, 0.2])
else:
    print(f"Target distribution: {model_df['was_targeted'].value_counts(normalize=True)}")

categorical_features = [
    'route_ran', 'route_category', 'coverage_assignment', 'receiver_type',
    'position_group', 'experience_level'
]
existing_cat_features = [col for col in categorical_features if col in model_df.columns]

print(f"One-hot encoding categorical features: {existing_cat_features}")
model_data = pd.get_dummies(model_df, columns=existing_cat_features, drop_first=True)

high_corr = ['age', 'target_distance', 'weight', 'receiver_type_not_receiver', 's_max', 'target_distance_calculated', 'time_in_pocket', 'height_inches', 'receiver_type_TE', 'red_zone']
model_data = model_data.drop(high_corr, axis=1)

print("Creating feature matrix and target vector...")
X = model_data.drop(['was_targeted', 'gameId', 'playId', 'nflId'], axis=1, errors='ignore')
y = model_data['was_targeted']

cols_with_missing = X.columns[X.isnull().any()].tolist()
if cols_with_missing:
    print(f"Removing {len(cols_with_missing)} columns with missing values")
    X = X.drop(cols_with_missing, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} examples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} examples, {X_test.shape[1]} features")


3. PREPARE TRAINING DATA
----------------------
Target distribution: was_targeted
0    0.800732
1    0.199268
Name: proportion, dtype: float64
One-hot encoding categorical features: ['route_ran', 'route_category', 'coverage_assignment', 'receiver_type', 'position_group', 'experience_level']
Creating feature matrix and target vector...
Training set: 36939 examples, 70 features
Test set: 9235 examples, 70 features


In [30]:
# 3.5 Run Pandas Profile Report to Find Highly Correlated Features
pp.ProfileReport(X_train, title="Training Data Profile Report").to_file("train_data_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 17.10it/s]


In [42]:
# 4. Finding Best Hyperparameters for Model
print("\n4. MODEL CROSS VALIDATION")
print("------------------------------")

scaler = StandardScaler()
scaler.fit(X_train)

scaled_X_train = scaler.transform(X_train)
scaled_X_test = scaler.transform(X_test)

params = {'n_estimators': [50, 75, 100, 125], 'max_depth': [5, 10, 20, 25], 'max_features': ['sqrt'], 'min_samples_split': [2, 5],
        'min_samples_leaf': [2], 'random_state': [42], 'class_weight': ['balanced']}

rf = RandomForestClassifier()
rf_cv = GridSearchCV(rf, params)

print("Implementing Cross Validation...")
train_start = time.time()
rf_cv.fit(scaled_X_train, y_train)
train_time = time.time() - train_start
print(f"Cross validation done in {train_time:.2f} seconds")

print("Best parameters: ")
print(rf_cv.best_params_)

print("Prediction using best found parameters...")
y_pred = rf_cv.predict(scaled_X_test)
y_prob = rf_cv.predict_proba(scaled_X_test)[:, 1]

accuracy = (y_pred == y_test).mean()
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nModel Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))


4. MODEL CROSS VALIDATION
------------------------------
Implementing Cross Validation...
Cross validation done in 668.59 seconds
Best parameters: 
{'class_weight': 'balanced', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 125, 'random_state': 42}
Prediction using best found parameters...

Model Performance:
Accuracy:  0.8942
Precision: 0.8071
Recall:    0.6163
ROC AUC:   0.9352

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      7395
           1       0.81      0.62      0.70      1840

    accuracy                           0.89      9235
   macro avg       0.86      0.79      0.82      9235
weighted avg       0.89      0.89      0.89      9235



In [5]:
# 4. MODEL TRAINING AND EVALUATION
print("\n4. MODEL TRAINING AND EVALUATION")
print("------------------------------")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        class_weight='balanced'
    ))
])

print("Training model...")
train_start = time.time()
pipeline.fit(X_train, y_train)
train_time = time.time() - train_start
print(f"Model trained in {train_time:.2f} seconds")

print("Evaluating model...")
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = (y_pred == y_test).mean()
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nModel Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

print("\nAnalyzing feature importance...")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': pipeline.named_steps['classifier'].feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Features:")
print(feature_importance.head(20).to_string(index=False))


4. MODEL TRAINING AND EVALUATION
------------------------------
Training model...
Model trained in 8.26 seconds
Evaluating model...

Model Performance:
Accuracy:  0.8933
Precision: 0.8021
Recall:    0.6168
ROC AUC:   0.9349

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      7395
           1       0.80      0.62      0.70      1840

    accuracy                           0.89      9235
   macro avg       0.86      0.79      0.82      9235
weighted avg       0.89      0.89      0.89      9235


Analyzing feature importance...

Top 20 Features:
                Feature  Importance
            pass_length    0.112072
movement_alignment_mean    0.106486
       s_mean_POST_SNAP    0.068923
           target_width    0.068146
        s_max_POST_SNAP    0.052631
          time_to_throw    0.050404
           target_depth    0.047454
                dis_sum    0.041917
                 s_mean    0.039463
    

In [18]:
# 4. MODEL TRAINING AND EVALUATION
print("\n4. MODEL TRAINING AND EVALUATION")
print("------------------------------")

top20 = feature_importance['Feature'].head(50)
X_top20 = X[top20]

X_train, X_test, y_train, y_test = train_test_split(
    X_top20, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        class_weight='balanced'
    ))
])

print("Training model...")
train_start = time.time()
pipeline.fit(X_train, y_train)
train_time = time.time() - train_start
print(f"Model trained in {train_time:.2f} seconds")

print("Evaluating model...")
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = (y_pred == y_test).mean()
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nModel Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))


4. MODEL TRAINING AND EVALUATION
------------------------------
Training model...
Model trained in 9.13 seconds
Evaluating model...

Model Performance:
Accuracy:  0.8419
Precision: 0.6112
Recall:    0.5600
ROC AUC:   0.8872

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90      7464
           1       0.61      0.56      0.58      1850

    accuracy                           0.84      9314
   macro avg       0.75      0.74      0.74      9314
weighted avg       0.84      0.84      0.84      9314



In [6]:
# 5. TARGET RECOMMENDATION FUNCTION
print("\n5. TARGET RECOMMENDATION FUNCTION")
print("------------------------------")

def recommend_target(game_id, play_id, model, data_df, players_df=None, confidence_threshold=0.4):
    play_data = data_df[(data_df['gameId'] == game_id) & (data_df['playId'] == play_id)].copy()

    if play_data.empty:
        return "No receivers found for this play"

    if 'displayName' in play_data.columns:
        player_names = play_data[['nflId', 'displayName']]
    elif players_df is not None and not players_df.empty:
        player_names = pd.merge(
            play_data[['nflId']],
            players_df[['nflId', 'displayName']],
            on='nflId',
            how='left'
        )
    else:
        player_names = pd.DataFrame({
            'nflId': play_data['nflId'],
            'displayName': play_data['nflId'].apply(lambda x: f"Player {x}")
        })

    features = play_data.drop(['was_targeted', 'gameId', 'playId', 'nflId'], axis=1, errors='ignore')

    missing_cols = set(X.columns) - set(features.columns)
    for col in missing_cols:
        features[col] = 0

    features = features[X.columns]

    play_data['target_probability'] = model.predict_proba(features)[:, 1]

    def get_recommendation(prob):
        if prob >= confidence_threshold:
            return "STRONG TARGET"
        elif prob >= confidence_threshold * 0.75:
            return "GOOD OPTION"
        elif prob >= confidence_threshold * 0.5:
            return "SECONDARY OPTION"
        else:
            return "LOW PRIORITY"

    play_data['recommendation'] = play_data['target_probability'].apply(get_recommendation)

    result = pd.merge(
        play_data[['nflId', 'target_probability', 'recommendation']],
        player_names,
        on='nflId',
        how='left'
    )

    context_cols = ['route_ran', 'route_category', 'coverage_assignment', 'route_depth_match',
                   'position_group', 'route_directness']
    available_context = [col for col in context_cols if col in play_data.columns]

    if available_context:
        result = pd.merge(
            result,
            play_data[['nflId'] + available_context],
            on='nflId',
            how='left'
        )

    if 'was_targeted' in play_data.columns:
        result['was_targeted'] = play_data['was_targeted']

    return result.sort_values('target_probability', ascending=False)

print("Testing recommendation function with diverse examples...")

bills_game_id = 2022090800

play_examples = [
    (bills_game_id, 56, "Red zone play"),
    (bills_game_id, 122, "Third down conversion attempt"),
    (bills_game_id, 167, "2-minute drill situation"),
]

for game_id, play_id, description in play_examples:
    print(f"\n=== {description} ===")
    print(f"Recommendations for Game {game_id}, Play {play_id}:")
    recommendations = recommend_target(
        game_id,
        play_id,
        pipeline,
        model_data,
        datasets['players'] if 'players' in datasets else None
    )

    if isinstance(recommendations, str):
        print(recommendations)
        continue

    display_cols = ['displayName', 'recommendation', 'target_probability']

    context_cols = ['route_ran', 'coverage_assignment', 'position_group', 'route_directness']
    display_cols.extend([col for col in context_cols if col in recommendations.columns])

    if 'was_targeted' in recommendations.columns:
        display_cols.append('was_targeted')

    print(recommendations[display_cols].head().to_string(index=False))

    if 'was_targeted' in recommendations.columns:
        actual_target = recommendations[recommendations['was_targeted'] == 1]
        if not actual_target.empty and not recommendations.empty:
            recommended_target = recommendations.iloc[0]

            if actual_target['nflId'].values[0] == recommended_target['nflId']:
                print("✓ Model recommendation matches the actual target!")
            else:
                print(f"× Model recommended a different target than actual.")

                if 'plays' in datasets and not datasets['plays'].empty:
                    play_info = datasets['plays'][
                        (datasets['plays']['gameId'] == game_id) &
                        (datasets['plays']['playId'] == play_id)
                    ]

                    if not play_info.empty:
                        sit_info = []
                        if 'down' in play_info.columns:
                            sit_info.append(f"Down: {play_info['down'].values[0]}")
                        if 'yardsToGo' in play_info.columns:
                            sit_info.append(f"Distance: {play_info['yardsToGo'].values[0]}")
                        if 'absoluteYardlineNumber' in play_info.columns:
                            yard_line = 100 - play_info['absoluteYardlineNumber'].values[0]
                            sit_info.append(f"Yard line: {yard_line}")

                        if sit_info:
                            print(f"Situation: {', '.join(sit_info)}")


5. TARGET RECOMMENDATION FUNCTION
------------------------------
Testing recommendation function with diverse examples...

=== Red zone play ===
Recommendations for Game 2022090800, Play 56:
    displayName recommendation  target_probability  route_directness  was_targeted
   Stefon Diggs  STRONG TARGET            0.747582          0.556191           1.0
Isaiah McKenzie  STRONG TARGET            0.433433          0.772786           0.0
    Dawson Knox    GOOD OPTION            0.318634          0.738127           0.0
    Dawson Knox    GOOD OPTION            0.318634          0.738127           0.0
    Dawson Knox    GOOD OPTION            0.318634          0.738127           0.0
✓ Model recommendation matches the actual target!

=== Third down conversion attempt ===
Recommendations for Game 2022090800, Play 122:
     displayName recommendation  target_probability  route_directness  was_targeted
Devin Singletary  STRONG TARGET            0.741557          0.889437           0.0
    St

In [7]:
# 6. MODEL INSIGHTS AND ANALYSIS
print("\n6. MODEL INSIGHTS AND ANALYSIS")
print("-----------------------------")

print("\nTargeting patterns by down:")
if 'down' in model_df.columns:
    down_analysis = model_df.groupby('down')['was_targeted'].mean().reset_index()
    down_analysis.columns = ['Down', 'Target Probability']
    print(down_analysis.to_string(index=False))

if all(col in model_df.columns for col in ['route_category', 'coverage_assignment']):
    route_coverage = model_df.groupby(['route_category', 'coverage_assignment'])['was_targeted'].mean().reset_index()
    route_coverage.columns = ['Route Type', 'Coverage', 'Target Probability']
    print("\nTarget probability by route and coverage:")
    print(route_coverage.sort_values('Target Probability', ascending=False).head(10).to_string(index=False))

if 'shotgun' in model_df.columns:
    formation_analysis = model_df.groupby('shotgun')['was_targeted'].mean().reset_index()
    formation_analysis.columns = ['Shotgun', 'Target Probability']
    print("\nTarget probability by formation (shotgun):")
    print(formation_analysis.to_string(index=False))

if 'experience_level' in model_df.columns:
    exp_analysis = model_df.groupby('experience_level')['was_targeted'].mean().reset_index()
    exp_analysis.columns = ['Experience Level', 'Target Probability']
    print("\nTarget probability by player experience:")
    print(exp_analysis.to_string(index=False))

print("\nCompletion probability by route type:")
if 'route_category' in model_df.columns:
    if 'plays' in datasets and 'passResult' in datasets['plays'].columns:
        completion_data = model_df.merge(
            datasets['plays'][['gameId', 'playId', 'passResult']],
            on=['gameId', 'playId'],
            how='left'
        )

        completion_data['completed'] = (completion_data['passResult'] == 'C').astype(int)
        route_analysis = completion_data.groupby('route_category')['completed'].mean().reset_index()
        route_analysis.columns = ['Route Type', 'Completion Rate']
        print(route_analysis.to_string(index=False))


6. MODEL INSIGHTS AND ANALYSIS
-----------------------------

Targeting patterns by down:
 Down  Target Probability
    1            0.204439
    2            0.199838
    3            0.190426
    4            0.213307

Target probability by route and coverage:
Route Type Coverage  Target Probability
   unknown       DF            1.000000
   unknown       3M            0.750000
     short       DF            0.700000
     short      4OL            0.346154
   unknown      HCL            0.336364
   unknown      4IR            0.333333
     short      HCL            0.314414
     short       3M            0.307692
   unknown      HCR            0.307339
    medium       DF            0.306122

Target probability by formation (shotgun):
 Shotgun  Target Probability
       0            0.205834
       1            0.196262

Target probability by player experience:
Experience Level  Target Probability
           prime            0.199026
    rookie_young            0.193932
          se

In [8]:
# 7. SUMMARY
print("\n7. SUMMARY")
print("---------")

total_time = time.time() - start_time
print(f"Total execution time: {total_time:.2f} seconds")

print("\nModel status:")
print(f"- Accuracy: {accuracy:.4f}")
print(f"- ROC AUC: {roc_auc:.4f}")
print(f"- Features used: {X.shape[1]}")
print(f"- Training examples: {X_train.shape[0]}")

if loading_errors:
    print(f"\nWarning: {len(loading_errors)} datasets could not be loaded properly")
    print("Some functionality may be limited")
else:
    print("\nAll data loaded successfully")

print("\nEnhanced QB Decision Recommendation System complete!")


7. SUMMARY
---------
Total execution time: 202.25 seconds

Model status:
- Accuracy: 0.8933
- ROC AUC: 0.9349
- Features used: 70
- Training examples: 36939

All data loaded successfully

Enhanced QB Decision Recommendation System complete!


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import glob
import time

print("QB Decision Recommendation System - Multi-Week Training")
print("====================================================")
start_time = time.time()

# 1. DATA LOADING - With Multi-Week Tracking Data
print("\n1. LOADING DATA")
print("--------------")

print("Loading core datasets...")
core_datasets = {
    'games': '../data/raw/games.csv',
    'plays': '../data/raw/plays.csv',
    'player_play': '../data/raw/player_play.csv',
    'players': '../data/raw/players.csv',
    'qb_features': '../data/processed/qb_decision_features.csv',
    'receiver_features': '../data/processed/receiver_target_features.csv',
    'player_features': '../data/processed/player_features.csv',
}

datasets = {}
for name, path in core_datasets.items():
    try:
        datasets[name] = pd.read_csv(path)
        print(f"✓ Loaded {name}: {len(datasets[name])} rows")
    except Exception as e:
        print(f"✗ Error loading {name}: {e}")
        datasets[name] = pd.DataFrame()

print("\nLoading tracking data from all available weeks...")
tracking_files = glob.glob('../data/tracking_week_*.csv')
tracking_dfs = []

for file_path in sorted(tracking_files):
    try:
        week_num = int(file_path.split('_')[-1].split('.')[0])
        print(f"Loading tracking data for Week {week_num}...", end='')
        df = pd.read_csv(file_path)
        df['week'] = week_num
        tracking_dfs.append(df)
        print(f" ✓ ({len(df)} rows)")
    except Exception as e:
        print(f" ✗ Error: {e}")

if tracking_dfs:
    print("Combining tracking data from all weeks...")
    all_tracking = pd.concat(tracking_dfs, ignore_index=True)
    print(f"✓ Combined tracking data: {len(all_tracking)} rows from {len(tracking_dfs)} weeks")
    datasets['tracking_all'] = all_tracking
else:
    print("✗ No tracking data could be loaded")
    datasets['tracking_all'] = pd.DataFrame()

# 2. DATA PREPROCESSING AND FEATURE ENGINEERING
print("\n2. DATA PREPROCESSING AND FEATURE ENGINEERING")
print("-------------------------------------------")

print("Preparing feature set...")
if 'receiver_features' in datasets and not datasets['receiver_features'].empty:
    model_df = datasets['receiver_features'].copy()
    print(f"Base feature set: {len(model_df)} receiver target examples")
else:
    print("ERROR: No receiver_features data available!")
    exit(1)

if not datasets['player_features'].empty:
    physical_attributes = [
        'height_inches', 'weight', 'receiver_type', 'bmi', 'height_vs_db',
        'position_group', 'age', 'is_receiver'
    ]
    
    available_attributes = [col for col in physical_attributes if col in datasets['player_features'].columns]
    
    if available_attributes:
        model_df = pd.merge(
            model_df,
            datasets['player_features'][['nflId'] + available_attributes],
            on='nflId',
            how='left'
        )
        print(f"Added player physical attributes: {available_attributes}")

if not datasets['qb_features'].empty:
    qb_features_list = [
        'pass_length', 'time_to_throw', 'pressure', 'expected_points',
        'time_in_pocket', 'play_action', 'shotgun', 'empty',
        'man_coverage', 'zone_coverage', 
        'epa_success', 'target_depth', 'target_width', 'target_distance'
    ]
    
    numeric_qb_cols = []
    for col in qb_features_list:
        if col in datasets['qb_features'].columns:
            if np.issubdtype(datasets['qb_features'][col].dtype, np.number) or datasets['qb_features'][col].dtype == bool:
                numeric_qb_cols.append(col)
    
    if numeric_qb_cols:
        qb_features_unique = datasets['qb_features'][['gameId', 'playId'] + numeric_qb_cols].drop_duplicates(['gameId', 'playId'])
        
        model_df = pd.merge(
            model_df,
            qb_features_unique,
            on=['gameId', 'playId'],
            how='left'
        )
        print(f"Added QB features: {numeric_qb_cols}")

if not datasets['tracking_all'].empty:
    print("Processing tracking data from all weeks...")
    
    tracking_presnap = datasets['tracking_all'][datasets['tracking_all']['frameType'] == 'BEFORE_SNAP']
    print(f"Pre-snap frames: {len(tracking_presnap)} rows")
    
    last_presnap_frames = tracking_presnap.sort_values(['gameId', 'playId', 'week', 'frameId']) \
        .groupby(['gameId', 'playId', 'week', 'nflId']).last().reset_index()
    print(f"Last pre-snap frames: {len(last_presnap_frames)} rows")
    
    tracking_features = []
    game_play_combos = model_df[['gameId', 'playId', 'nflId']].drop_duplicates()
    
    print(f"Calculating tracking features for {len(game_play_combos)} game-play-player combinations...")
    for i, (game_id, play_id, nfl_id) in enumerate(game_play_combos.values):
        player_tracking = last_presnap_frames[
            (last_presnap_frames['gameId'] == game_id) & 
            (last_presnap_frames['playId'] == play_id) & 
            (last_presnap_frames['nflId'] == nfl_id)
        ]
        
        if player_tracking.empty:
            continue
        
        avg_speed = player_tracking['s'].mean() if 's' in player_tracking.columns else np.nan
        max_speed = player_tracking['s'].max() if 's' in player_tracking.columns else np.nan
        avg_accel = player_tracking['a'].mean() if 'a' in player_tracking.columns else np.nan
        
        nearest_defender_dist = np.nan
        
        if 'team' in player_tracking.columns:
            player_team = player_tracking['team'].iloc[0]
            
            for week, week_data in player_tracking.groupby('week'):
                play_tracking = last_presnap_frames[
                    (last_presnap_frames['gameId'] == game_id) & 
                    (last_presnap_frames['playId'] == play_id) & 
                    (last_presnap_frames['week'] == week)
                ]
                
                defenders = play_tracking[play_tracking['team'] != player_team]
                
                if not defenders.empty:
                    x = week_data['x'].iloc[0]
                    y = week_data['y'].iloc[0]
                    distances = np.sqrt((defenders['x'] - x)**2 + (defenders['y'] - y)**2)
                    
                    min_dist = distances.min()
                    if np.isnan(nearest_defender_dist) or min_dist < nearest_defender_dist:
                        nearest_defender_dist = min_dist
        
        tracking_features.append({
            'gameId': game_id,
            'playId': play_id,
            'nflId': nfl_id,
            'avg_speed': avg_speed,
            'max_speed': max_speed,
            'avg_accel': avg_accel,
            'nearest_defender_dist': nearest_defender_dist
        })
        
        if (i+1) % 1000 == 0 or i+1 == len(game_play_combos):
            print(f"Processed {i+1}/{len(game_play_combos)} combinations...")
    
    tracking_features_df = pd.DataFrame(tracking_features)
    
    if not tracking_features_df.empty:
        model_df = pd.merge(
            model_df,
            tracking_features_df,
            on=['gameId', 'playId', 'nflId'],
            how='left'
        )
        print(f"Added multi-week tracking features: {tracking_features_df.columns.tolist()[3:]}")

print("Creating derived features...")

available_columns = model_df.columns.tolist()

if 'route_ran' in available_columns:
    route_categories = {
        'FLAT': 'short',
        'HITCH': 'short',
        'OUT': 'short',
        'SCREEN': 'short',
        'SLANT': 'short',
        'CROSS': 'medium',
        'DIG': 'medium', 
        'IN': 'medium',
        'POST': 'deep',
        'CORNER': 'deep',
        'GO': 'deep'
    }
    model_df['route_category'] = model_df['route_ran'].map(route_categories)
    print("  Added route_category")

if all(col in available_columns for col in ['down', 'yards_to_go']):
    model_df['third_down_critical'] = ((model_df['down'] == 3) & 
                                     (model_df['yards_to_go'] >= 3) & 
                                     (model_df['yards_to_go'] <= 7)).astype(int)
    print("  Added third_down_critical")

if all(col in available_columns for col in ['yards_from_goal', 'down']):
    model_df['red_zone'] = (model_df['yards_from_goal'] <= 20).astype(int)
    model_df['redzone_scoring_opportunity'] = ((model_df['red_zone'] == 1) & 
                                             (model_df['down'] >= 3)).astype(int)
    print("  Added redzone_scoring_opportunity")

if all(col in available_columns for col in ['pressure', 'time_to_throw']):
    model_df['quick_throw_under_pressure'] = ((model_df['pressure'] == 1) & 
                                            (model_df['time_to_throw'] < 2.5)).astype(int)
    print("  Added quick_throw_under_pressure")

if all(col in available_columns for col in ['route_category', 'yards_to_go']):
    def depth_match(row):
        if pd.isna(row['route_category']):
            return np.nan
        
        if row['yards_to_go'] <= 3 and row['route_category'] == 'short':
            return 1
        elif 3 < row['yards_to_go'] <= 10 and row['route_category'] == 'medium':
            return 1
        elif row['yards_to_go'] > 10 and row['route_category'] == 'deep':
            return 1
        else:
            return 0
    
    model_df['route_depth_match'] = model_df.apply(depth_match, axis=1)
    print("  Added route_depth_match")

if 'age' in available_columns:
    def get_experience_level(age):
        if pd.isna(age):
            return 'unknown'
        elif age <= 24:
            return 'rookie_young'
        elif age <= 28:
            return 'prime'
        elif age <= 32:
            return 'veteran'
        else:
            return 'senior'
    
    model_df['experience_level'] = model_df['age'].apply(get_experience_level)
    
    if 'third_down_critical' in available_columns:
        model_df['veteran_in_critical'] = (
            (model_df['experience_level'].isin(['veteran', 'senior'])) & 
            (model_df['third_down_critical'] == 1)
        ).astype(int)
        print("  Added veteran_in_critical")
    
    print("  Added experience_level")

if all(col in available_columns for col in ['coverage_assignment', 'height_inches']):
    model_df['height_advantage_in_man'] = (
        (model_df['coverage_assignment'] == 'MAN') & 
        (model_df['height_vs_db'] > 2)
    ).astype(int)
    print("  Added height_advantage_in_man")

if all(col in available_columns for col in ['shotgun', 'route_category']):
    model_df['shotgun_deep_route'] = (
        (model_df['shotgun'] == 1) & 
        (model_df['route_category'] == 'deep')
    ).astype(int)
    print("  Added shotgun_deep_route")

if all(col in available_columns for col in ['max_speed', 'nearest_defender_dist']):
    model_df['speed_with_separation'] = (
        (model_df['max_speed'] > model_df['max_speed'].median()) & 
        (model_df['nearest_defender_dist'] > 3)
    ).astype(int)
    print("  Added speed_with_separation")

print("Handling missing values...")

numeric_cols = model_df.select_dtypes(include=['float64', 'int64']).columns
model_df[numeric_cols] = model_df[numeric_cols].fillna(model_df[numeric_cols].median())

categorical_cols = model_df.select_dtypes(include=['object']).columns
model_df[categorical_cols] = model_df[categorical_cols].fillna('unknown')

print(f"Final dataset shape: {model_df.shape}")

# 3. PREPARE TRAINING DATA
print("\n3. PREPARE TRAINING DATA")
print("----------------------")

if 'was_targeted' not in model_df.columns:
    print("ERROR: Target variable 'was_targeted' not found")
    exit(1)
else:
    print(f"Target distribution: {model_df['was_targeted'].value_counts(normalize=True)}")

categorical_features = [
    'route_ran', 'route_category', 'coverage_assignment', 'receiver_type',
    'position_group', 'experience_level'
]
existing_cat_features = [col for col in categorical_features if col in model_df.columns]

print(f"One-hot encoding categorical features: {existing_cat_features}")
model_data = pd.get_dummies(model_df, columns=existing_cat_features, drop_first=True)

print("Creating feature matrix and target vector...")
X = model_data.drop(['was_targeted', 'gameId', 'playId', 'nflId'], axis=1, errors='ignore')
y = model_data['was_targeted']

cols_with_missing = X.columns[X.isnull().any()].tolist()
if cols_with_missing:
    print(f"Removing {len(cols_with_missing)} columns with missing values")
    X = X.drop(cols_with_missing, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} examples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} examples, {X_test.shape[1]} features")

# 4. MODEL TRAINING AND EVALUATION
print("\n4. MODEL TRAINING AND EVALUATION")
print("------------------------------")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        class_weight='balanced'
    ))
])

print("Training model...")
train_start = time.time()
pipeline.fit(X_train, y_train)
train_time = time.time() - train_start
print(f"Model trained in {train_time:.2f} seconds")

print("Evaluating model...")
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = (y_pred == y_test).mean()
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nModel Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

print("\nAnalyzing feature importance...")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': pipeline.named_steps['classifier'].feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Features:")
print(feature_importance.head(20).to_string(index=False))

# 5. TARGET RECOMMENDATION FUNCTION
print("\n5. TARGET RECOMMENDATION FUNCTION")
print("------------------------------")

def recommend_target(game_id, play_id, model, data_df, players_df=None, confidence_threshold=0.4):
    play_data = data_df[(data_df['gameId'] == game_id) & (data_df['playId'] == play_id)].copy()
    
    if play_data.empty:
        return "No receivers found for this play"
    
    if 'displayName' in play_data.columns:
        player_names = play_data[['nflId', 'displayName']]
    elif players_df is not None and not players_df.empty:
        player_names = pd.merge(
            play_data[['nflId']], 
            players_df[['nflId', 'displayName']],
            on='nflId',
            how='left'
        )
    else:
        player_names = pd.DataFrame({
            'nflId': play_data['nflId'],
            'displayName': play_data['nflId'].apply(lambda x: f"Player {x}")
        })
    
    features = play_data.drop(['was_targeted', 'gameId', 'playId', 'nflId'], axis=1, errors='ignore')
    
    missing_cols = set(X.columns) - set(features.columns)
    for col in missing_cols:
        features[col] = 0
    
    features = features[X.columns]
    
    play_data['target_probability'] = model.predict_proba(features)[:, 1]
    
    def get_recommendation(prob):
        if prob >= confidence_threshold:
            return "STRONG TARGET"
        elif prob >= confidence_threshold * 0.75:
            return "GOOD OPTION"
        elif prob >= confidence_threshold * 0.5:
            return "SECONDARY OPTION"
        else:
            return "LOW PRIORITY"
    
    play_data['recommendation'] = play_data['target_probability'].apply(get_recommendation)
    
    result = pd.merge(
        play_data[['nflId', 'target_probability', 'recommendation']], 
        player_names,
        on='nflId',
        how='left'
    )
    
    context_cols = ['route_ran', 'route_category', 'coverage_assignment', 'route_depth_match', 'position_group']
    available_context = [col for col in context_cols if col in play_data.columns]
    
    if available_context:
        result = pd.merge(
            result,
            play_data[['nflId'] + available_context],
            on='nflId',
            how='left'
        )
    
    if 'was_targeted' in play_data.columns:
        result['was_targeted'] = play_data['was_targeted']
    
    return result.sort_values('target_probability', ascending=False)

print("Testing recommendation function with diverse examples...")

play_examples = [
    (2022090800, 56, "Bills-Rams Red zone play"),
    (2022090800, 122, "Bills-Rams Third down conversion"),
    (2022091100, 45, "First quarter play from different game"),
    (2022091800, 201, "Late game situation")
]

for game_id, play_id, description in play_examples:
    print(f"\n=== {description} ===")
    print(f"Recommendations for Game {game_id}, Play {play_id}:")
    recommendations = recommend_target(
        game_id, 
        play_id, 
        pipeline, 
        model_data, 
        datasets['players']
    )
    
    if isinstance(recommendations, str):
        print(recommendations)
        continue
    
    display_cols = ['displayName', 'recommendation', 'target_probability']
    
    context_cols = ['route_ran', 'coverage_assignment', 'position_group']
    display_cols.extend([col for col in context_cols if col in recommendations.columns])
    
    if 'was_targeted' in recommendations.columns:
        display_cols.append('was_targeted')
    
    print(recommendations[display_cols].head().to_string(index=False))
    
    if 'was_targeted' in recommendations.columns:
        actual_target = recommendations[recommendations['was_targeted'] == 1]
        if not actual_target.empty and not recommendations.empty:
            recommended_target = recommendations.iloc[0]
            
            if actual_target['nflId'].values[0] == recommended_target['nflId']:
                print("✓ Model recommendation matches the actual target!")
            else:
                print(f"× Model recommended a different target than actual.")
                
                if not datasets['plays'].empty:
                    play_info = datasets['plays'][
                        (datasets['plays']['gameId'] == game_id) & 
                        (datasets['plays']['playId'] == play_id)
                    ]
                    
                    if not play_info.empty:
                        sit_info = []
                        if 'down' in play_info.columns:
                            sit_info.append(f"Down: {play_info['down'].values[0]}")
                        if 'yardsToGo' in play_info.columns:
                            sit_info.append(f"Distance: {play_info['yardsToGo'].values[0]}")
                        if 'absoluteYardlineNumber' in play_info.columns:
                            yard_line = 100 - play_info['absoluteYardlineNumber'].values[0]
                            sit_info.append(f"Yard line: {yard_line}")
                        
                        if sit_info:
                            print(f"Situation: {', '.join(sit_info)}")

# 6. MODEL INSIGHTS AND ANALYSIS
print("\n6. MODEL INSIGHTS AND ANALYSIS")
print("-----------------------------")

print("\nTargeting patterns by down:")
if 'down' in model_df.columns:
    down_analysis = model_df.groupby('down')['was_targeted'].mean().reset_index()
    down_analysis.columns = ['Down', 'Target Probability']
    print(down_analysis.to_string(index=False))

if all(col in model_df.columns for col in ['route_category', 'coverage_assignment']):
    route_coverage = model_df.groupby(['route_category', 'coverage_assignment'])['was_targeted'].mean().reset_index()
    route_coverage.columns = ['Route Type', 'Coverage', 'Target Probability']
    print("\nTarget probability by route and coverage:")
    print(route_coverage.sort_values('Target Probability', ascending=False).head(10).to_string(index=False))

if 'experience_level' in model_df.columns:
    exp_analysis = model_df.groupby('experience_level')['was_targeted'].mean().reset_index()
    exp_analysis.columns = ['Experience Level', 'Target Probability']
    print("\nTarget probability by player experience:")
    print(exp_analysis.to_string(index=False))

print("\nCompletion probability by route type:")
if 'route_category' in model_df.columns:
    if not datasets['plays'].empty and 'passResult' in datasets['plays'].columns:
        completion_data = model_df.merge(
            datasets['plays'][['gameId', 'playId', 'passResult']],
            on=['gameId', 'playId'],
            how='left'
        )
        
        completion_data['completed'] = (completion_data['passResult'] == 'C').astype(int)
        route_analysis = completion_data.groupby('route_category')['completed'].mean().reset_index()
        route_analysis.columns = ['Route Type', 'Completion Rate']
        print(route_analysis.to_string(index=False))

# 7. SUMMARY
print("\n7. SUMMARY")
print("---------")

total_time = time.time() - start_time
print(f"Total execution time: {total_time:.2f} seconds")

print("\nModel status:")
print(f"- Accuracy: {accuracy:.4f}")
print(f"- ROC AUC: {roc_auc:.4f}")
print(f"- Features used: {X.shape[1]}")
print(f"- Training examples: {X_train.shape[0]}")

print("\nMulti-week QB Decision Recommendation System complete!")

QB Decision Recommendation System - Multi-Week Training

1. LOADING DATA
--------------
Loading core datasets...
✓ Loaded games: 136 rows
✓ Loaded plays: 16124 rows
✓ Loaded player_play: 354727 rows
✓ Loaded players: 1697 rows
✓ Loaded qb_features: 9736 rows
✓ Loaded receiver_features: 46566 rows
✓ Loaded player_features: 1697 rows

Loading tracking data from all available weeks...
✗ No tracking data could be loaded

2. DATA PREPROCESSING AND FEATURE ENGINEERING
-------------------------------------------
Preparing feature set...
Base feature set: 46566 receiver target examples
Added player physical attributes: ['height_inches', 'weight', 'receiver_type', 'bmi', 'height_vs_db', 'position_group', 'age', 'is_receiver']
Added QB features: ['pass_length', 'time_to_throw', 'expected_points', 'time_in_pocket', 'play_action', 'shotgun', 'empty', 'epa_success', 'target_depth', 'target_width', 'target_distance']
Creating derived features...
  Added route_category
  Added third_down_critical
  A